In [1]:
import pandas as pd 

In [2]:
df=pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.isnull().sum()
df.shape

(50000, 2)

In [4]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

In [5]:

#Preprocessing 
df["review"]=df["review"].str.lower()

# Preprocessing 

In [6]:
import re 

1. Remove URL

In [7]:
def remove_url(text):
    text=re.sub(r"https\S+","",text)
    return text
df["review"]=df["review"].apply(remove_url)

2. Remove Punctuations

In [8]:
def remove_punctuations(text):
    text=re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
df["review"]=df["review"].apply(remove_punctuations)

3. Removing HTML

In [9]:
def remove_html(text):
    text=re.sub(r"<.*?>","",text)
    return text
df["review"]=df["review"].apply(remove_html)

4. Removing Stopwords

In [10]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sroy9\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sroy9\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sroy9\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [12]:
def remove_stopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text=text.replace(word,"")
    return text
df["review"]=df["review"].apply(remove_stopwords)


In [13]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


5. Stemming 

In [14]:
from nltk.stem import PorterStemmer

In [15]:
def stemming(text):
    ps=PorterStemmer()
    tokens=word_tokenize(text)
    stemmed_words=[]

    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"]=df["review"].apply(stemming)


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



6. Vectorization 

In [16]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


7. Encoding

In [17]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

In [22]:
y=df["sentiment"]

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df["review"])


In [25]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057136 stored elements and shape (49582, 5000)>

# Dataset and Dataloaders

In [24]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [27]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [26]:
X_train=X_train.toarray()
X_test=X_test.toarray()

In [30]:
train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [32]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)